# s99 MDD evidence pack (fts_c, nofts_c)

READ ONLY. For each model in `config.mdd.models` this notebook (1) inventories its reports, run summaries and registry fields, (2) copies existing report tables verbatim into named sheets, each with a header row naming the source file and sheet, and (3) lists every missing, empty or stale item in a `gaps` sheet. No metric is recomputed. The only writes are under `<config.mdd.doc_output_root>/<config.mdd.output_folder>/evidence/` (`mdd_evidence_<key>.xlsx`, `inventory.json`, `run_summaries/s99_mdd_evidence_pack.json`). No MLflow run and no write to `reports/run_summaries/`: the request restricts writes to `evidence/`.

In [0]:
# The project has no requirements.lock yet (same note as s06b); this module only needs openpyxl + pyyaml.
%pip install openpyxl pyyaml --quiet
dbutils.library.restartPython()

In [0]:
# s99_mdd_evidence_pack: MDD evidence pack. READ ONLY except files under <mdd.doc_output_root>/<mdd.output_folder>/evidence/.
import sys, os, json, hashlib, shutil, tempfile, re, csv, glob, importlib
from datetime import datetime, timezone
sys.path.insert(0, os.getcwd())            # notebook lives in src/
import common
importlib.reload(common)
import openpyxl
from openpyxl.cell.cell import ILLEGAL_CHARACTERS_RE

PROJECT_ROOT = common.find_project_root(os.getcwd())
cfg = common.load_config(os.path.join(PROJECT_ROOT, "config", "config.yaml"))
MDD = cfg["mdd"]
MODULE = "s99_mdd_evidence_pack"
RUN_TIME = datetime.now(timezone.utc).strftime("%Y-%m-%dT%H:%M:%S")
GIT = common.git_commit(PROJECT_ROOT)
try:
    BRANCH = common.git_branch(PROJECT_ROOT)
except Exception as e:                      # branch is informational only
    BRANCH = f"unknown ({type(e).__name__})"
CFG_SHA = common.config_hash(cfg)

DOC_ROOT = MDD["doc_output_root"].rstrip("/")
EVID = os.path.join(DOC_ROOT, MDD["output_folder"].strip("/"), "evidence")
EVID_REAL = os.path.realpath(EVID)

MODELS = []
for m in MDD["models"]:
    m = dict(m)
    m["artifacts_root"] = m["artifacts_root"].rstrip("/")
    assert os.path.isdir(m["artifacts_root"]), f"artifacts_root not found for {m['key']}: {m['artifacts_root']}"
    assert os.path.exists(m["registry"]), f"registry not found for {m['key']}: {m['registry']}"
    assert m["model"] in ("A", "B", "C"), f"{m['key']}: model must be A/B/C, got {m['model']}"
    MODELS.append(m)
assert {m["key"] for m in MODELS} == {"fts_c", "nofts_c"}, f"expected fts_c and nofts_c in config.mdd.models, got {[m['key'] for m in MODELS]}"

print(f"Module      : {MODULE} | run {RUN_TIME} UTC | branch {BRANCH} | git {GIT} | config {CFG_SHA[:12]}")
print(f"Evidence out: {EVID}")
for m in MODELS:
    print(f"  {m['key']:8s} model {m['model']} | {m['label']}\n           registry {m['registry']}\n           root     {m['artifacts_root']}")

In [0]:
def sha256_file(p, chunk=1 << 20):
    h = hashlib.sha256()
    with open(p, "rb") as f:
        for b in iter(lambda: f.read(chunk), b""):
            h.update(b)
    return h.hexdigest()

def mtime_utc(p):
    return datetime.fromtimestamp(os.stat(p).st_mtime, tz=timezone.utc).replace(tzinfo=None)

def read_json(p):
    with open(p) as f:
        return json.load(f)

def under_evidence(p):
    rp = os.path.realpath(p)
    return rp == EVID_REAL or rp.startswith(EVID_REAL + os.sep)

def snapshot_files(roots):
    """path -> (size, mtime_ns) for every file under the roots, excluding evidence/."""
    out = {}
    for r in roots:
        for dp, dn, fn in os.walk(r):
            if under_evidence(dp):
                dn[:] = []
                continue
            for f in fn:
                p = os.path.join(dp, f)
                st = os.stat(p)
                out[os.path.realpath(p)] = (st.st_size, st.st_mtime_ns)
    return out

def verify_locked_files(reg):
    """Re-hash every file recorded in registry.files; return list of mismatching keys."""
    bad = []
    for k, v in reg.get("files", {}).items():
        if not os.path.exists(v["path"]) or sha256_file(v["path"]) != v["sha256"]:
            bad.append(k)
    return bad

def parse_lock(s):
    # registry times are written by the cluster in UTC without offset (checked: s04_lock.json mtime = locked_at)
    return datetime.fromisoformat(str(s)) if s else None

WATCH_ROOTS = sorted({m["artifacts_root"] for m in MODELS} | {DOC_ROOT})
SNAP_BEFORE = snapshot_files(WATCH_ROOTS)
REG = {}
for m in MODELS:
    reg = read_json(m["registry"])
    REG[m["key"]] = {"reg": reg, "sha_before": sha256_file(m["registry"]), "bad_before": verify_locked_files(reg)}
    print(f"{m['key']}: registry {reg.get('version')} locked={reg.get('locked')} at {reg.get('locked_at')} | "
          f"{len(reg.get('files', {}))} locked files, {len(REG[m['key']]['bad_before'])} hash mismatches | sha {REG[m['key']]['sha_before'][:12]}")
print(f"Read-only snapshot: {len(SNAP_BEFORE)} files under {WATCH_ROOTS}")
WRITTEN = []          # every file this notebook writes

In [0]:
# Each item: section, sheet name, source kind, file (relative to the chosen root; glob allowed), sheet(s), root, stage, applies.
#   kind : xlsx | csv | json | registry (path in registry) | config (path in config.yaml) | project (file in the Git folder) | search (regex on report file names)
#   root : own = the model's artifacts_root; shared = main model root (E2/E3 are shared with the variant by design);
#          variant = the no-FTS variant root (FTS comparison lives there)
#   stage: pre = produced before the lock (E2-E12); post = produced at/after E13 (must not be older than the lock)
#   applies: all | variant | main
def I(section, name, kind, file=None, sheet=None, root="own", stage="pre", applies="all", path=None, note=""):
    assert len(name) <= 31, name
    return dict(section=section, name=name, kind=kind, file=file, sheet=sheet, root=root, stage=stage,
                applies=applies, path=path, note=note)

OOT = "reports/05_oot.xlsx"; TR = "reports/04_training.xlsx"; TU = "reports/04_tuning.xlsx"
CUT = "reports/cutoff/cutoff_analysis_v1.0.1*_A_C.xlsx"
ITEMS = [
    # data (E2/E3 are shared inputs: read from the main model root for the variant)
    I("data", "data_filter_waterfall", "xlsx", "reports/01_exclusion_waterfall.xlsx", "Sheet1", "shared"),
    I("data", "data_confirmed_counts_cfg", "config", path=["expected_counts"], note="confirmed counts as set in config.yaml (ALL, DEV, DEV_EARLY, DEV_LATE, OOT)"),
    I("data", "data_counts_by_window", "xlsx", "reports/02_split.xlsx", "window_summary", "shared"),
    I("data", "data_dev_early_late", "xlsx", "reports/02_split.xlsx", "dev_early_late", "shared"),
    I("data", "data_status_by_window", "xlsx", "reports/02_split.xlsx", "window_x_status", "shared"),
    I("data", "data_tfold_stats", "xlsx", "reports/02_split.xlsx", "t_fold_stats", "shared"),
    I("data", "data_active_reporting_xtab", "xlsx", "reports/01_active_reporting_xtab.xlsx", "Sheet1", "shared"),
    # target
    I("target", "target_vintage_monthly", "xlsx", "reports/01_vintage_monthly.xlsx", "Sheet1", "shared"),
    I("target", "target_roll_rate", "search", r"roll", root="shared", note="roll-rate summary"),
    # funnel
    I("funnel", "fun_by_layer", "xlsx", "reports/03_funnel_summary.xlsx", "by_layer"),
    I("funnel", "fun_by_source", "xlsx", "reports/03_funnel_summary.xlsx", "by_source"),
    I("funnel", "fun_by_layer_source", "xlsx", "reports/03_funnel_summary.xlsx", "by_layer_source"),
    I("funnel", "fun_drop_rules", "xlsx", "reports/03_funnel_summary.xlsx", "drop_rules"),
    I("funnel", "L0_counts_by_reason", "xlsx", "reports/03_L0.xlsx", "counts_by_reason"),
    I("funnel", "L0_counts_by_source", "xlsx", "reports/03_L0.xlsx", "counts_by_source"),
    I("funnel", "L0_feature_detail", "xlsx", "reports/03_L0.xlsx", "feature_detail"),
    I("funnel", "L0_scope_excl_prefix", "xlsx", "reports/03_L0.xlsx", "scope_excl_prefix", applies="variant"),
    I("funnel", "L1_counts_by_reason", "xlsx", "reports/03_L1.xlsx", "counts_by_reason"),
    I("funnel", "L1_iv_table", "xlsx", "reports/03_L1.xlsx", "feature_detail"),
    I("funnel", "L2_periods", "xlsx", "reports/03_L2.xlsx", "periods"),
    I("funnel", "L2_period_results", "xlsx", "reports/03_L2.xlsx", "feature_detail"),
    I("funnel", "L3_psi_summary", "xlsx", "reports/03_psi.xlsx", "summary"),
    I("funnel", "L3_red_features", "xlsx", "reports/03_psi.xlsx", "red_features"),
    I("funnel", "L3_psi_decisions_todo", "xlsx", "reports/03_psi.xlsx", "psi_decisions_todo"),
    I("funnel", "L3_psi_decisions_file", "project", "config/psi_decisions.csv", note="human decision file as on this branch now (not hash-recorded in the registry)"),
    I("funnel", "L4_pool_summary", "xlsx", "reports/03_L4.xlsx", "summary"),
    I("funnel", "L4_vif", "xlsx", "reports/03_L4.xlsx", "vif_table"),
    I("funnel", "L4_corr_drops", "xlsx", "reports/03_L4.xlsx", "corr_drops"),
    I("funnel", "L4_pool_json", "json", "feature_lists/l4_pool.json"),
    I("funnel", "L5_selection_info", "xlsx", "reports/03_L5.xlsx", "selection_info"),
    I("funnel", "L5_select_freq", "xlsx", "reports/03_L5.xlsx", "select_freq"),
    I("funnel", "L5_logit_steps", "xlsx", "reports/03_L5.xlsx", "logit_steps"),
    I("funnel", "L5_logit_summary", "xlsx", "reports/03_L5.xlsx", "logit_summary"),
    I("funnel", "L5_points_concentration", "xlsx", "reports/03_L5.xlsx", "points_concentration"),
    I("funnel", "L5_final_list", "xlsx", "reports/03_L5.xlsx", "final_list"),
    # models
    I("models", "mod_tuning_choice", "xlsx", TU, "choice"),
    I("models", "mod_fixed_params", "xlsx", TU, "fixed_params"),
    I("models", "mod_C_shortlist", "xlsx", TU, "C_shortlist"),
    I("models", "mod_B_shortlist", "xlsx", TU, "B_shortlist"),
    I("models", "mod_C_tfold", "xlsx", TU, "C_tfold"),
    I("models", "mod_chosen_params", "json", "params/chosen_params.json"),
    I("models", "mod_lgb_rounds", "registry", path=["lightgbm_rounds"]),
    I("models", "mod_A_points", "xlsx", TR, "A_points"),
    I("models", "mod_A_coefficients", "xlsx", TR, "A_coefficients"),
    I("models", "mod_oof_metrics", "xlsx", TR, "oof_metrics"),
    I("models", "mod_oof_fold_metrics", "xlsx", TR, "oof_fold_metrics"),
    I("models", "mod_tfold_metrics", "xlsx", TR, "tfold_metrics"),
    I("models", "mod_tfold_summary", "xlsx", TR, "tfold_summary"),
    I("models", "mod_C_importance", "xlsx", TR, "C_importance"),
    I("models", "mod_B_shape_check", "xlsx", TR, "B_shape_check"),
    I("models", "mod_training_notes", "xlsx", TR, "notes"),
    I("models", "mod_reg_dev_oof", "registry", path=["dev_oof_metrics"]),
    I("models", "mod_reg_tfold", "registry", path=["t_fold"]),
    I("models", "mod_reg_ebm_shape", "registry", path=["ebm_shape_check"]),
    # OOT (E13) incl. A/B/C selection evidence
    I("oot", "oot_summary", "xlsx", OOT, "summary", stage="post"),
    I("oot", "oot_rules_R1_R4", "xlsx", OOT, "rules", stage="post"),
    I("oot", "oot_counts", "xlsx", OOT, "counts", stage="post"),
    I("oot", "oot_metrics", "xlsx", OOT, "metrics", stage="post"),
    I("oot", "oot_paired_bootstrap", "xlsx", OOT, "paired_bootstrap", stage="post"),
    I("oot", "oot_delong", "xlsx", OOT, "delong", stage="post"),
    I("oot", "oot_bands", "xlsx", OOT, "bands", stage="post"),
    I("oot", "oot_band_monotonic", "xlsx", OOT, "band_monotonic", stage="post"),
    I("oot", "oot_decile_csv", "csv", "reports/decile_analysis_oot.csv", stage="post"),
    I("oot", "oot_calibration", "xlsx", OOT, "calibration", stage="post"),
    I("oot", "oot_calibration_by_decile", "xlsx", OOT, "calibration_by_decile", stage="post"),
    I("oot", "oot_score_psi", "xlsx", OOT, "score_psi", stage="post"),
    I("oot", "oot_segments", "xlsx", OOT, "segments", stage="post"),
    I("oot", "oot_watch_approval", "xlsx", OOT, "approval_by_watch", stage="post"),
    I("oot", "oot_L6_summary", "xlsx", OOT, "L6_summary", stage="post"),
    I("oot", "oot_diagnosis", "xlsx", OOT, "diagnosis", stage="post"),
    I("oot", "oot_vs_main_v101", "xlsx", OOT, "vs_main_v101", stage="post", applies="variant"),
    I("oot", "oot_reg_evaluation", "registry", path=["oot_evaluation"], stage="post"),
    I("oot", "oot_run_summary", "json", "reports/run_summaries/s05_oot.json", stage="post"),
    I("oot", "backtest", "search", r"backtest", stage="post", note="E14 back-test report (05_backtest*.xlsx / s05_backtest.json)"),
    # calibration and scaling
    I("calib", "cal_platt", "xlsx", OOT, "platt", stage="post"),
    I("calib", "cal_platt_file", "json", "calibration/platt_dev_oof.json", stage="post"),
    I("calib", "cal_scaling_A", "json", "models/model_spec.json", path=["A", "scaling"]),
    I("calib", "cal_factor_offset_A", "json", "models/model_spec.json", path=["A"], note="keys factor / offset"),
    I("calib", "cal_capped_shares", "search", r"(06_calib|calibrat.*\.xlsx|capp)", stage="post", note="E15 capped-score shares"),
    # cut-off, reject analysis, FTS comparison, variant consistency
    I("cutoff", "cut_summary", "xlsx", CUT, "summary", stage="post"),
    I("cutoff", "cut_grid_oot_A", "xlsx", CUT, "grid_oot_A", stage="post"),
    I("cutoff", "cut_grid_oot_C", "xlsx", CUT, "grid_oot_C", stage="post"),
    I("cutoff", "cut_grid_dev_oof_A", "xlsx", CUT, "grid_dev_oof_A", stage="post"),
    I("cutoff", "cut_grid_dev_oof_C", "xlsx", CUT, "grid_dev_oof_C", stage="post"),
    I("cutoff", "cut_C_vs_A", "xlsx", CUT, "C_vs_A", stage="post"),
    I("cutoff", "cut_score_bands_C", "xlsx", CUT, "score_bands_C", stage="post"),
    I("ri", "ri_summary", "xlsx", "reports/ri/ri_reject_analysis.xlsx", "summary", stage="post", applies="variant"),
    I("ri", "ri_intake_waterfall", "xlsx", "reports/ri/ri_reject_analysis.xlsx", "intake_waterfall", stage="post", applies="variant"),
    I("ri", "ri_fts_summary", "xlsx", "reports/ri_fts/ri_reject_fts_compare.xlsx", "summary", stage="post", applies="main"),
    I("ri", "ri_fts_universe_counts", "xlsx", "reports/ri_fts/ri_reject_fts_compare.xlsx", "universe_counts", stage="post", applies="main"),
    I("fts_cmp", "fts_cmp_performance", "csv", "reports/05c_fts_comparison_performance.csv", root="variant", stage="post"),
    I("fts_cmp", "fts_cmp_paired_bootstrap", "csv", "reports/05c_fts_comparison_paired_bootstrap.csv", root="variant", stage="post"),
    I("fts_cmp", "fts_cmp_counts", "csv", "reports/05c_fts_comparison_counts.csv", root="variant", stage="post"),
    I("fts_cmp", "fts_cmp_net_bads_ci", "csv", "reports/05c_fts_comparison_net_bads_bootstrap_ci.csv", root="variant", stage="post"),
    I("fts_cmp", "ascore_cmp_performance", "csv", "reports/05d_ascore_comparison_performance.csv", root="variant", stage="post"),
    I("fts_cmp", "ascore_cmp_paired_boot", "csv", "reports/05d_ascore_comparison_paired_bootstrap.csv", root="variant", stage="post"),
    I("variant", "vc_summary", "xlsx", "reports/99_variant_consistency.xlsx", "summary", applies="variant"),
    I("variant", "vc_column_summary", "xlsx", "reports/99_variant_consistency.xlsx", "2_column_summary", applies="variant"),
    I("variant", "vc_mismatches", "xlsx", "reports/99_variant_consistency.xlsx", "2_mismatches", applies="variant"),
    I("variant", "vc_l4_l5_differences", "xlsx", "reports/99_variant_consistency.xlsx", "3_l4_l5_differences", applies="variant"),
    I("variant", "vc_funnel_hash", "xlsx", "reports/99_variant_consistency.xlsx", "4_funnel_hash", applies="variant"),
    I("variant", "vc_run_summary", "json", "reports/run_summaries/s99_variant_consistency.json", applies="variant"),
    # reason codes
    I("reason", "rc_map_used", "project", cfg["review_files"]["reason_code_map"], note="config.review_files.reason_code_map as on this branch now (not hash-recorded in the registry)"),
    I("reason", "rc_draft_run_summary", "json", "reports/run_summaries/s05_reason_code_map_draft.json"),
    I("reason", "rc_group_frequencies", "search", r"(reason_code_freq|reason_group_freq|rc_freq|18_reason)", stage="post", note="reason-code group frequencies (E18)"),
]
assert len({i["name"] for i in ITEMS}) == len(ITEMS), "duplicate sheet names in catalogue"
print(f"{len(ITEMS)} catalogue items; sections: {sorted({i['section'] for i in ITEMS})}")

In [0]:
def is_variant_reg(reg):
    return "variant_comparison" in reg.get("pre_registered", {})

MAIN = [m for m in MODELS if not is_variant_reg(REG[m["key"]]["reg"])]
VAR = [m for m in MODELS if is_variant_reg(REG[m["key"]]["reg"])]
assert len(MAIN) == 1 and len(VAR) == 1, f"expected one main and one variant registry, got main {[m['key'] for m in MAIN]} variant {[m['key'] for m in VAR]}"
MAIN, VAR = MAIN[0], VAR[0]
REGISTERED_MAIN = REG[VAR["key"]]["reg"]["pre_registered"]["variant_comparison"].get("main_registry")

for m in MODELS:
    m["is_variant"] = m is VAR
    m["roots"] = {"own": m["artifacts_root"],
                  "shared": MAIN["artifacts_root"],        # E2/E3 shared inputs live in the main root
                  "variant": VAR["artifacts_root"]}
    print(f"{m['key']}: {'variant' if m['is_variant'] else 'main'} | shared root {m['roots']['shared']}")
print(f"Variant registry's registered main_registry: {REGISTERED_MAIN} (exists: {os.path.exists(REGISTERED_MAIN) if REGISTERED_MAIN else None})")

In [0]:
def rget(d, *ks):
    for k in ks:
        if not isinstance(d, dict) or k not in d:
            return None
        d = d[k]
    return d

def find_keys(obj, pattern, prefix=""):
    """All (path, value) in a nested dict whose key matches the regex (used to look for snapshot/extraction fields)."""
    out = []
    if isinstance(obj, dict):
        for k, v in obj.items():
            p = f"{prefix}.{k}" if prefix else str(k)
            if k != "files" and re.search(pattern, str(k), re.I):
                out.append((p, v))
            if k != "files":
                out += find_keys(v, pattern, p)
    return out

def list_reports(root):
    rows, base = [], os.path.join(root, "reports")
    for dp, dn, fn in os.walk(base):
        dn.sort()
        if under_evidence(dp):
            dn[:] = []
            continue
        for f in sorted(fn):
            p = os.path.join(dp, f)
            st = os.stat(p)
            rows.append({"folder": os.path.relpath(dp, root), "name": f, "size_bytes": st.st_size,
                         "modified_utc": mtime_utc(p).isoformat(timespec="seconds"), "path": p})
    return rows

GAPS = {m["key"]: [] for m in MODELS}
def gap(m, section, item, gap_type, detail, expected=""):
    GAPS[m["key"]].append({"model_key": m["key"], "section": section, "item": item, "gap_type": gap_type,
                           "detail": detail, "expected_source": expected})

INV = {}
for m in MODELS:
    reg = REG[m["key"]]["reg"]
    snap = find_keys(reg, r"(snapshot_table|raw_snapshot|extraction_date)")
    flog = find_keys(reg, r"feature_log")
    cfg_flog = cfg["tables"]["feature_log"] if m["is_variant"] else rget(cfg, "main_tables", "feature_log")
    fields = {
        "version": reg.get("version"), "locked": reg.get("locked"), "locked_at": reg.get("locked_at"),
        "created_at": reg.get("created_at"), "lock_run": reg.get("lock_run"), "git_commit": reg.get("git_commit"),
        "config_sha256": reg.get("config_sha256"), "funnel_config_hash": reg.get("funnel_config_hash"),
        "data_snapshot_fields_in_registry": {p: v for p, v in snap} or None,
        "signoff": reg.get("signoff"),
        "selected_model": rget(reg, "oot_evaluation", "selected_model"),
        "selection_status": rget(reg, "oot_evaluation", "status"),
        "selection_reason": rget(reg, "oot_evaluation", "selection_reason"),
        "pre_registered": reg.get("pre_registered"),
        "oot_evaluation": reg.get("oot_evaluation"),
        "production": reg.get("production"),
        "lightgbm_rounds": reg.get("lightgbm_rounds"), "code_versions": reg.get("code_versions"),
        "source_runs": reg.get("source_runs"), "notes": reg.get("notes"),
        "n_locked_files": len(reg.get("files", {})),
        "locked_file_hash_mismatches": REG[m["key"]]["bad_before"],
    }
    reference = {   # NOT registry values: current config on this branch, recorded so the MDD author knows where they came from
        "data_raw_snapshot_table (config.data, current)": rget(cfg, "data", "raw_snapshot_table"),
        "data_extraction_date (config.data, current)": rget(cfg, "data", "extraction_date"),
        "feature_log_table (registry)": {p: v for p, v in flog} or None,
        "feature_log_table (config, current)": f"{cfg['databricks']['catalog']}.{cfg['databricks']['schema']}.{cfg_flog}" if cfg_flog else None,
    }
    sel = fields["selected_model"]
    flag = None if sel == m["model"] else f"registry selected model {sel} ({fields['selection_status']}) is NOT the model documented in config.mdd.models ({m['model']})"
    # Selection override (option 1): a human decision recorded in config.mdd.models[key].override. The registry is never
    # changed; the rule-based selection stays as evidence and the override is disclosed next to it.
    OV_REQUIRED = ("model", "approved_by", "role", "date", "reason", "evidence_ref")
    ov_raw = m.get("override")
    OV = None
    if ov_raw:
        ov_missing = [k for k in OV_REQUIRED if not str(ov_raw.get(k) or "").strip()]
        try:
            datetime.fromisoformat(str(ov_raw.get("date")))
        except Exception:
            if "date" not in ov_missing:
                ov_missing.append("date (not an ISO date)")
        ev = str(ov_raw.get("evidence_ref") or "").strip()
        ev_is_path = ev.startswith("/")
        ev_exists = os.path.exists(ev) if ev_is_path else None
        OV = {**{k: ov_raw.get(k) for k in ov_raw}, "missing_fields": ov_missing,
              "evidence_ref_is_path": ev_is_path, "evidence_ref_exists": ev_exists,
              "evidence_ref_sha256": sha256_file(ev) if ev_is_path and ev_exists and os.path.isfile(ev) else None,
              "source": f"config/config.yaml mdd.models[key={m['key']}].override"}
        OV["valid"] = (not ov_missing) and str(ov_raw.get("model")) == m["model"] and ev_exists is not False
    basis = "RULE" if sel == m["model"] else ("APPROVED_OVERRIDE" if OV and OV["valid"] else "UNSUPPORTED")
    INV[m["key"]] = {"key": m["key"], "label": m["label"], "mdd_model": m["model"], "role": "variant" if m["is_variant"] else "main",
                     "registry": m["registry"], "registry_sha256": REG[m["key"]]["sha_before"],
                     "artifacts_root": m["artifacts_root"], "shared_inputs_root": m["roots"]["shared"],
                     "selected_model": sel, "selected_status": fields["selection_status"],
                     "selected_is_mdd_model": sel == m["model"], "selected_model_flag": flag,
                     "documented_basis": basis, "override": OV,
                     "registry_fields": fields, "reference_not_in_registry": reference,
                     "report_files": list_reports(m["artifacts_root"])}
    # registry-level gaps
    if flag and basis == "UNSUPPORTED":
        why = ("; no override recorded in config.mdd.models[].override" if not OV else
               f"; override recorded but not valid (missing {OV['missing_fields']}, override model {OV.get('model')}, evidence exists {OV['evidence_ref_exists']})")
        gap(m, "oot", "selected_model", "SELECTED_MODEL_MISMATCH", flag + why, "model_registry.json oot_evaluation.selected_model")
    if OV and not OV["valid"]:
        gap(m, "oot", "selection_override", "OVERRIDE_INCOMPLETE",
            f"missing/invalid fields {OV['missing_fields']}; override model {OV.get('model')} vs documented {m['model']}; evidence_ref exists {OV['evidence_ref_exists']}",
            OV["source"])
    if OV and sel == m["model"]:
        gap(m, "oot", "selection_override", "NOTE", "override recorded although the rule-based selection already equals the documented model", OV["source"])
    if fields["selection_status"] != "FINAL":
        gap(m, "oot", "selection_status", "SELECTION_NOT_FINAL", f"oot_evaluation.status = {fields['selection_status']}; reason: {fields['selection_reason']}", "model_registry.json")
    if not fields["locked"]:
        gap(m, "registry", "locked", "NOT_LOCKED", "registry locked is not true", "model_registry.json")
    for k in ("git_commit",):
        if str(fields[k] or "unknown") == "unknown":
            gap(m, "registry", k, "UNKNOWN_VALUE", f"registry {k} = {fields[k]!r}; base commit cannot be evidenced", "model_registry.json")
    if not snap:
        gap(m, "registry", "data_snapshot_table / extraction_date", "NOT_IN_REGISTRY",
            f"registry has no data snapshot table or extraction date; config (current) says {reference['data_raw_snapshot_table (config.data, current)']} / {reference['data_extraction_date (config.data, current)']}", "model_registry.json")
    if not flog:
        gap(m, "registry", "feature_log_table", "NOT_IN_REGISTRY", f"registry does not record the feature-log table; config (current) = {reference['feature_log_table (config, current)']}", "model_registry.json")
    if fields["production"] is None:
        gap(m, "registry", "production", "NOT_IN_REGISTRY", "no production section (E16+ not run or not recorded)", "model_registry.json production")
    for k in ("decision_rules_signed_by", "signed_date", "model_owner"):
        if not str(rget(reg, "signoff", k) or "").strip():
            gap(m, "registry", f"signoff.{k}", "EMPTY", "sign-off field empty", "model_registry.json signoff")
    if fields["locked_file_hash_mismatches"]:
        gap(m, "registry", "files", "LOCKED_FILE_HASH_MISMATCH", f"{len(fields['locked_file_hash_mismatches'])} locked files differ: {fields['locked_file_hash_mismatches']}", "model_registry.json files")
    if m["is_variant"] and REGISTERED_MAIN and not os.path.exists(REGISTERED_MAIN):
        gap(m, "registry", "pre_registered.variant_comparison.main_registry", "REGISTERED_PATH_MISSING",
            f"registered path {REGISTERED_MAIN} does not exist; evidence uses config.mdd main registry {MAIN['registry']}", "model_registry.json")
    print(f"{m['key']}: {len(INV[m['key']]['report_files'])} report files | version {fields['version']} locked {fields['locked']} {fields['locked_at']} "
          f"| selected {sel} {fields['selection_status']} | documented {m['model']} basis {basis} | flag: {flag}")

In [0]:
import datetime as _dt
WB_CACHE, SHA_CACHE = {}, {}
def open_wb(p):
    if p not in WB_CACHE:
        WB_CACHE[p] = openpyxl.load_workbook(p, read_only=True, data_only=True)
    return WB_CACHE[p]

def fsha(p):
    if p not in SHA_CACHE:
        SHA_CACHE[p] = sha256_file(p)
    return SHA_CACHE[p]

def xlsx_rows(p, sheet):
    rows = [list(r) for r in open_wb(p)[sheet].iter_rows(values_only=True)]
    while rows and all(v is None for v in rows[-1]):
        rows.pop()
    return rows

def csv_rows(p):
    with open(p, newline="", encoding="utf-8-sig") as f:
        return [r for r in csv.reader(f)]

def flatten(obj, prefix=""):
    if isinstance(obj, dict):
        if not obj:
            return [[prefix, "{}"]]
        out = []
        for k, v in obj.items():
            out += flatten(v, f"{prefix}.{k}" if prefix else str(k))
        return out
    if isinstance(obj, list):
        if not obj or all(not isinstance(x, (dict, list)) for x in obj):
            return [[prefix, json.dumps(obj, default=str)]]
        out = []
        for i, v in enumerate(obj):
            out += flatten(v, f"{prefix}[{i}]")
        return out
    return [[prefix, obj]]

def has_data(rows, header_rows=1):
    return any(any(v not in (None, "") for v in r) for r in rows[header_rows:])

def resolve(root, pattern):
    return sorted(p for p in glob.glob(os.path.join(root, pattern)) if os.path.isfile(p))

def search_reports(root, regex):
    out = []
    for dp, dn, fn in os.walk(os.path.join(root, "reports")):
        if under_evidence(dp):
            dn[:] = []
            continue
        out += [os.path.join(dp, f) for f in fn if re.search(regex, f, re.I)]
    return sorted(out)

PACK = {}   # model key -> list of copied sheets {name, header, rows, index-row}
for m in MODELS:
    reg = REG[m["key"]]["reg"]
    lock = parse_lock(reg.get("locked_at"))
    sheets, index = [], []

    def add(item, rows, src, sub, header_rows=1, sheet_name=None, check_time=True):
        name = sheet_name or item["name"]
        is_file = os.path.isfile(src)
        mt = mtime_utc(src) if is_file else None
        sha = fsha(src) if is_file else None
        empty = not has_data(rows, header_rows)
        status = "EMPTY" if empty else "COPIED"
        header = [[f"SOURCE: {src} | {sub}"],
                  [f"source modified_utc: {mt.isoformat(timespec='seconds') if mt else '-'} | sha256: {sha or '-'} | copied verbatim by "
                   f"{MODULE} at {RUN_TIME} UTC; no values recomputed" + (f" | note: {item['note']}" if item["note"] else "")],
                  []]
        sheets.append({"name": name, "header": header, "rows": rows, "src": src, "sub": sub, "kind": item["kind"]})
        index.append({"sheet": name, "section": item["section"], "item": item["name"], "kind": item["kind"], "source_path": src,
                      "source_part": sub, "source_modified_utc": mt.isoformat(timespec="seconds") if mt else None,
                      "source_sha256": sha, "rows_copied": len(rows), "status": status, "note": item["note"]})
        if empty:
            gap(m, item["section"], name, "EMPTY", f"{sub} in {src} has no data rows", src)
        if check_time and mt and lock:
            if item["stage"] == "post" and mt < lock:
                gap(m, item["section"], name, "OLDER_THAN_LOCK", f"source modified {mt:%Y-%m-%d %H:%M:%S} UTC < registry lock {lock}", src)
            if item["stage"] == "pre" and mt > lock:
                gap(m, item["section"], name, "MODIFIED_AFTER_LOCK", f"pre-lock artefact modified {mt:%Y-%m-%d %H:%M:%S} UTC > registry lock {lock} (check it is the version used by the lock)", src)

    for it in ITEMS:
        if it["applies"] == "variant" and not m["is_variant"]:
            continue
        if it["applies"] == "main" and m["is_variant"]:
            continue
        root = m["roots"][it["root"]]
        k = it["kind"]
        if k == "registry":
            v = rget(reg, *it["path"])
            if v is None:
                gap(m, it["section"], it["name"], "NOT_IN_REGISTRY", f"registry field {'.'.join(it['path'])} absent", m["registry"])
                continue
            add(it, [["field", "value"]] + flatten(v, ".".join(it["path"])), m["registry"], f"registry field: {'.'.join(it['path'])}", check_time=False)
        elif k == "config":
            v = rget(cfg, *it["path"])
            src = os.path.join(PROJECT_ROOT, "config", "config.yaml")
            if v is None:
                gap(m, it["section"], it["name"], "MISSING", f"config key {'.'.join(it['path'])} absent", src)
                continue
            add(it, [["key", "value"]] + flatten(v, ".".join(it["path"])), src, f"config key: {'.'.join(it['path'])} (current file)", check_time=False)
        elif k == "project":
            src = os.path.join(PROJECT_ROOT, it["file"])
            if not os.path.exists(src):
                gap(m, it["section"], it["name"], "MISSING_FILE", f"{it['file']} not found", src)
                continue
            add(it, csv_rows(src), src, "whole file (csv)", check_time=False)
        elif k == "search":
            hits = sorted(set(search_reports(root, it["file"]) + search_reports(m["artifacts_root"], it["file"])))
            if not hits:
                gap(m, it["section"], it["name"], "MISSING_FILE", f"no report found matching /{it['file']}/ under {root}/reports"
                    + ("" if root == m["artifacts_root"] else f" or {m['artifacts_root']}/reports") + f" ({it['note']})", f"{root}/reports")
                continue
            for j, p in enumerate(hits):
                if p.endswith(".xlsx"):
                    for s in open_wb(p).sheetnames:
                        add(it, xlsx_rows(p, s), p, f"sheet: {s}", sheet_name=f"{it['name'][:24]}_{j}_{len(sheets)}"[:31])
                elif p.endswith(".csv"):
                    add(it, csv_rows(p), p, "whole file (csv)", sheet_name=f"{it['name'][:26]}_{j}"[:31])
                elif p.endswith(".json"):
                    add(it, [["field", "value"]] + flatten(read_json(p)), p, "whole file (json)", sheet_name=f"{it['name'][:26]}_{j}"[:31])
                else:
                    gap(m, it["section"], it["name"], "NOT_TABULAR", f"found {p} but it is not xlsx/csv/json; not copied", p)
        else:
            hits = resolve(root, it["file"])
            if not hits:
                gap(m, it["section"], it["name"], "MISSING_FILE", f"{it['file']} not found under {root}", os.path.join(root, it["file"]))
                continue
            if len(hits) > 1:
                gap(m, it["section"], it["name"], "AMBIGUOUS_FILE", f"{len(hits)} files match {it['file']}: {hits}; none copied", root)
                continue
            p = hits[0]
            if k == "xlsx":
                if it["sheet"] not in open_wb(p).sheetnames:
                    gap(m, it["section"], it["name"], "MISSING_SHEET", f"sheet {it['sheet']!r} not in {p}", p)
                    continue
                add(it, xlsx_rows(p, it["sheet"]), p, f"sheet: {it['sheet']}")
            elif k == "csv":
                add(it, csv_rows(p), p, "whole file (csv)")
            elif k == "json":
                doc = read_json(p)
                v = rget(doc, *it["path"]) if it["path"] else doc
                if v is None:
                    gap(m, it["section"], it["name"], "MISSING", f"key {'.'.join(it['path'])} absent in {p}", p)
                    continue
                sub = f"json key: {'.'.join(it['path'])}" if it["path"] else "whole file (json)"
                rows = [["field", "value"]] + flatten(v, ".".join(it["path"] or []))
                if it["name"] == "cal_factor_offset_A":     # only the scalar keys, the rest is copied elsewhere
                    rows = [["field", "value"]] + [[f"A.{kk}", v[kk]] for kk in ("factor", "offset") if kk in v]
                    if len(rows) == 1:
                        gap(m, it["section"], it["name"], "MISSING", f"model_spec A has no factor/offset", p)
                        continue
                add(it, rows, p, sub)

    # explicit content checks (presence only, nothing recomputed)
    w = next((s for s in sheets if s["name"] == "data_counts_by_window"), None)
    if w is None or not any(str(r[0]).strip().upper() == "ALL" for r in w["rows"][1:] if r):
        gap(m, "data", "confirmed_counts.ALL", "NOT_IN_REPORT", "no ALL row in any data report; ALL is only available from config.expected_counts (copied as data_confirmed_counts_cfg)", "reports/02_split.xlsx")
    if m["is_variant"]:
        newer = resolve(m["artifacts_root"], "reports/09_variant_consistency.xlsx")
        old = resolve(m["artifacts_root"], "reports/99_variant_consistency.xlsx")
        if newer and old and mtime_utc(newer[0]) > mtime_utc(old[0]):
            gap(m, "variant", "vc_summary", "NOTE", f"a newer 09_variant_consistency.xlsx ({mtime_utc(newer[0]):%Y-%m-%d %H:%M}) exists next to the requested 99_variant_consistency.xlsx ({mtime_utc(old[0]):%Y-%m-%d %H:%M}); only 99_ copied", newer[0])
    gap_rc = rget(reg, "oot_evaluation")
    if "reason_code_map" not in json.dumps(reg.get("files", {})) and "reason_code_map" not in json.dumps(gap_rc or {}):
        gap(m, "reason", "rc_map_used", "NOT_IN_REGISTRY", "registry does not record which reason_code_map file/hash was used; rc_map_used is the current branch file", m["registry"])
    if not rget(cfg, "strategy", "proposed_cutoff"):
        gap(m, "cutoff", "strategy.proposed_cutoff", "NOT_SET", "config.strategy.proposed_cutoff is null: no approval rate at an agreed cut-off is reported (grid only)", "config.yaml")
    PACK[m["key"]] = {"sheets": sheets, "index": index}
    print(f"{m['key']}: {len(sheets)} sheets copied ({sum(i['status']=='EMPTY' for i in index)} empty) | {len(GAPS[m['key']])} gaps")

In [0]:
def safe(v):
    if v is None or isinstance(v, (bool, int, _dt.datetime, _dt.date, _dt.time)):
        return v
    if isinstance(v, float):
        return v if v == v and v not in (float("inf"), float("-inf")) else str(v)
    s = v if isinstance(v, str) else (json.dumps(v, default=str) if isinstance(v, (dict, list)) else str(v))
    return ILLEGAL_CHARACTERS_RE.sub("", s)

def write_rows(ws, rows):
    for r in rows:
        ws.append([safe(v) for v in r])
        if r:
            for c in ws[ws.max_row]:
                if isinstance(c.value, str) and c.value.startswith("="):
                    c.data_type = "s"          # keep text, never a formula

def write_out(local, dest):
    assert under_evidence(dest), f"refusing to write outside evidence/: {dest}"
    os.makedirs(os.path.dirname(dest), exist_ok=True)
    shutil.copyfile(local, dest)                    # copyfile, not copy2 (UC Volume utime issue)
    WRITTEN.append(dest)

TMP = tempfile.mkdtemp(prefix="s99_mdd_")
OUT_WB = {}
for m in MODELS:
    key, inv, pk = m["key"], INV[m["key"]], PACK[m["key"]]
    wb = openpyxl.Workbook()
    wb.remove(wb.active)
    ws = wb.create_sheet("meta")
    write_rows(ws, [["module", "run_time_utc", "git_commit", "git_branch", "config_sha256", "model_key", "label", "mdd_model",
                     "registry", "registry_sha256", "registry_version", "locked_at", "selected_model", "selected_status", "selected_model_flag",
                     "documented_basis", "override_approved_by", "override_date", "n_copied_sheets", "n_gaps"],
                    [MODULE, RUN_TIME, GIT, BRANCH, CFG_SHA, key, m["label"], m["model"], m["registry"], inv["registry_sha256"],
                     inv["registry_fields"]["version"], inv["registry_fields"]["locked_at"], inv["selected_model"], inv["selected_status"],
                     inv["selected_model_flag"] or "", inv["documented_basis"], rget(inv, "override", "approved_by") or "",
                     str(rget(inv, "override", "date") or ""), len(pk["sheets"]), len(GAPS[key])]])
    ws = wb.create_sheet("index")
    cols = ["sheet", "section", "item", "kind", "source_path", "source_part", "source_modified_utc", "source_sha256", "rows_copied", "status", "note"]
    write_rows(ws, [cols] + [[r[c] for c in cols] for r in pk["index"]])
    ws = wb.create_sheet("inventory_reports")
    write_rows(ws, [[f"SOURCE: directory listing of {m['artifacts_root']}/reports (recursive, incl. run_summaries/)"], [],
                    ["folder", "name", "size_bytes", "modified_utc", "path"]] +
               [[r["folder"], r["name"], r["size_bytes"], r["modified_utc"], r["path"]] for r in inv["report_files"]])
    ws = wb.create_sheet("registry_fields")
    write_rows(ws, [[f"SOURCE: {m['registry']} (registry fields) + reference values marked 'not in registry'"], [], ["field", "value"]]
               + flatten(inv["registry_fields"]) + [["--- reference, NOT in registry ---", None]] + flatten(inv["reference_not_in_registry"]))
    ws = wb.create_sheet("selection_override")
    write_rows(ws, [[f"SOURCE: {m['registry']} (oot_evaluation, rule-based) + config/config.yaml mdd.models[key={key}].override (human decision)"], [],
                    ["field", "value"],
                    ["rule_selected_model", inv["selected_model"]], ["rule_status", inv["selected_status"]],
                    ["rule_selection_reason", inv["registry_fields"]["selection_reason"]],
                    ["rule_evidence_sheets", "oot_rules_R1_R4, oot_paired_bootstrap, oot_metrics, oot_reg_evaluation"],
                    ["documented_model", m["model"]], ["documented_basis", inv["documented_basis"]]]
               + (flatten(inv["override"], "override") if inv["override"] else [["override", "none recorded"]]))
    for s in pk["sheets"]:
        write_rows(wb.create_sheet(s["name"]), s["header"] + s["rows"])
    ws = wb.create_sheet("gaps")
    gcols = ["model_key", "section", "item", "gap_type", "detail", "expected_source"]
    write_rows(ws, [gcols] + [[g[c] for c in gcols] for g in GAPS[key]])
    local = os.path.join(TMP, f"mdd_evidence_{key}.xlsx")
    wb.save(local)
    dest = os.path.join(EVID, f"mdd_evidence_{key}.xlsx")
    write_out(local, dest)
    OUT_WB[key] = dest
    inv["evidence_workbook"] = dest
    inv["n_copied_sheets"] = len(pk["sheets"])
    inv["n_gaps"] = len(GAPS[key])
    inv["gaps_by_type"] = {t: sum(g["gap_type"] == t for g in GAPS[key]) for t in sorted({g["gap_type"] for g in GAPS[key]})}
    print(f"{key}: wrote {dest} ({len(wb.sheetnames)} sheets)")

INVENTORY = {"module": MODULE, "run_time_utc": RUN_TIME, "git_commit": GIT, "git_branch": BRANCH, "config_sha256": CFG_SHA,
             "mdd": {k: MDD[k] for k in ("doc_output_root", "output_folder")}, "evidence_folder": EVID,
             "variant_registered_main_registry": REGISTERED_MAIN,
             "models": [INV[m["key"]] for m in MODELS]}
local = os.path.join(TMP, "inventory.json")
with open(local, "w") as f:
    json.dump(INVENTORY, f, indent=2, default=str)
write_out(local, os.path.join(EVID, "inventory.json"))
for m in MODELS:
    print(f"{m['key']}: selected {INV[m['key']]['selected_model']} ({INV[m['key']]['selected_status']}) | flag: {INV[m['key']]['selected_model_flag']} | gaps {INV[m['key']]['gaps_by_type']}")

In [0]:
SNAP_AFTER = snapshot_files(WATCH_ROOTS)
changed = sorted(p for p in set(SNAP_BEFORE) | set(SNAP_AFTER) if SNAP_BEFORE.get(p) != SNAP_AFTER.get(p))
reg_after = {m["key"]: sha256_file(m["registry"]) for m in MODELS}
bad_after = {m["key"]: verify_locked_files(REG[m["key"]]["reg"]) for m in MODELS}

# re-open the written workbooks and check structure + verbatim copy (written cells vs a fresh read of the source)
src_named, verbatim_bad, meta_first, gaps_present = True, [], True, True
for m in MODELS:
    wb = openpyxl.load_workbook(OUT_WB[m["key"]], read_only=True, data_only=True)
    meta_first &= wb.sheetnames[0] == "meta"
    gaps_present &= "gaps" in wb.sheetnames and "selection_override" in wb.sheetnames
    for s in PACK[m["key"]]["sheets"]:
        got = [list(r) for r in wb[s["name"]].iter_rows(values_only=True)]
        a1 = str(got[0][0] or "")
        src_named &= a1.startswith("SOURCE: ") and s["src"] in a1
        if s["kind"] in ("xlsx", "csv", "project", "search"):
            fresh = xlsx_rows(s["src"], s["sub"].split("sheet: ", 1)[1]) if s["src"].endswith(".xlsx") else csv_rows(s["src"])
            body = got[3:]
            exp = [[safe(v) for v in r] for r in fresh]
            norm = lambda rows: [[(None if v == "" else v) for v in r] for r in rows]
            trim = lambda rows: [r[:max([i + 1 for i, v in enumerate(r) if v is not None] or [0])] for r in rows]
            if trim(norm(body)) != trim(norm(exp)):
                verbatim_bad.append(f"{m['key']}:{s['name']}")

inv_check = read_json(os.path.join(EVID, "inventory.json"))
inv_ok = all(x["selected_model"] is not None and (x["selected_model_flag"] is not None) == (x["selected_model"] != x["mdd_model"])
             for x in inv_check["models"]) and {x["key"] for x in inv_check["models"]} == {"fts_c", "nofts_c"}

def basis_ok(x):
    gtypes = {g["gap_type"] for g in GAPS[x["key"]] if g["item"] == "selected_model"}
    if x["selected_model"] == x["mdd_model"]:
        return x["documented_basis"] == "RULE"
    if x["documented_basis"] == "APPROVED_OVERRIDE":
        return bool(x["override"] and x["override"]["valid"]) and "SELECTED_MODEL_MISMATCH" not in gtypes
    return x["documented_basis"] == "UNSUPPORTED" and "SELECTED_MODEL_MISMATCH" in gtypes

checks = {
    "No file written or changed outside evidence/ (before/after snapshot of artifacts roots + doc_output_root)": not changed,
    "Every file written is under evidence/": all(under_evidence(p) for p in WRITTEN),
    "One workbook per model (fts_c, nofts_c) + inventory.json written": set(OUT_WB) == {"fts_c", "nofts_c"} and os.path.exists(os.path.join(EVID, "inventory.json")),
    "Registry files unchanged (sha256 before = after)": all(reg_after[k] == REG[k]["sha_before"] for k in reg_after),
    "Locked file hash verification unchanged (before = after)": all(bad_after[k] == REG[k]["bad_before"] for k in bad_after),
    "Every copied sheet names its source in A1": src_named,
    "Copied xlsx/csv sheets equal a fresh read of the source (verbatim)": not verbatim_bad,
    "First sheet is meta; gaps and selection_override sheets present": meta_first and gaps_present,
    "Documented model basis: RULE iff registry selection = mdd model, else APPROVED_OVERRIDE (valid override) or UNSUPPORTED (+ mismatch gap)":
        all(basis_ok(x) for x in inv_check["models"]),
    "inventory.json records each selected model and flags it iff it is not the config.mdd model": inv_ok,
}
for k, v in checks.items():
    print(("PASS " if v else "FAIL ") + k)
if changed: print("changed outside evidence:", changed[:20])
if verbatim_bad: print("not verbatim:", verbatim_bad)

summary = {"module": MODULE, "run_time_utc": RUN_TIME, "git_commit": GIT, "git_branch": BRANCH, "config_hash": CFG_SHA,
           "outputs": WRITTEN + [os.path.join(EVID, "run_summaries", f"{MODULE}.json")],
           "models": {m["key"]: {"selected_model": INV[m["key"]]["selected_model"], "selected_status": INV[m["key"]]["selected_status"],
                                 "flag": INV[m["key"]]["selected_model_flag"], "n_copied_sheets": INV[m["key"]]["n_copied_sheets"],
                                 "gaps_by_type": INV[m["key"]]["gaps_by_type"]} for m in MODELS},
           "checks": checks, "pass_fail": "PASS" if all(checks.values()) else "FAIL"}
local = os.path.join(TMP, f"{MODULE}.json")
with open(local, "w") as f:
    json.dump(summary, f, indent=2, default=str)
write_out(local, os.path.join(EVID, "run_summaries", f"{MODULE}.json"))

for k, v in checks.items():
    assert v, f"Acceptance failed: {k}"
print("All acceptance checks PASS")